In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import ( accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tqdm.keras import TqdmCallback
import matplotlib.pyplot as plt

In [ ]:
def create_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,), name="Input")
    encoded = Dense(
        4,
        activation="relu",
        activity_regularizer=regularizers.l1(10e-5),
        name="Encoding_1",
    )(input_layer)
    latent = Dense(2, activation="relu", name="Latent")(encoded)
    decoded = Dense(4, activation="relu", name="Decoding_2")(latent)
    output_layer = Dense(input_dim, activation="linear", name="Output")(decoded)
    autoencoder = Model(input_layer, output_layer)
    return autoencoder

def predict(model, data, threshold):
    reconstructions = model.predict(data)
    loss = tf.keras.losses.mae(reconstructions, data)
    return (loss.numpy() < threshold).astype(int)

def print_stats(predictions, labels):
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    print(f"Accuracy = {accuracy}")
    print(f"Precision = {precision}")
    print(f"Recall = {recall}")
    print(f"F1 score = {f1}")
    return accuracy, precision, recall, f1

In [ ]:
data = pd.read_csv("Data/data.csv")
X = data.drop(columns=["Label"])
y = data["Label"]

In [5]:
data["Label"].value_counts()

Label
1    181880
0      1275
Name: count, dtype: int64

In [ ]:
X = X[['365', '101', '86', '100', '130']]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
y_test.value_counts()

Label
1    36365
0      266
Name: count, dtype: int64

In [ ]:
autoencoder = create_autoencoder(input_dim=n_features)
autoencoder.compile(optimizer="adadelta", loss="mse")
autoencoder.summary()

# Fit the model
history = autoencoder.fit(
    X_train,
    X_train,
    batch_size=64,
    epochs=10,
    verbose=0,
    validation_split=0.15,
    callbacks=[TqdmCallback(), EarlyStopping(patience=3)],
)

# Predict reconstruction errors for the training set
reconstructions = autoencoder.predict(X_train)
train_loss = tf.keras.losses.mae(reconstructions, X_train).numpy()
threshold = np.mean(train_loss) + np.std(train_loss)
print("Threshold: ", threshold)

# Predict reconstruction errors for the test set
reconstructions = autoencoder.predict(X_test)
test_loss = tf.keras.losses.mae(reconstructions, X_test).numpy()
preds = predict(autoencoder, X_test, threshold)

accuracy, precision, recall, f1 = print_stats(preds, y_test)
conf_matrix = confusion_matrix(y_test, preds)
print(conf_matrix)

In [15]:
# Lưu lại mô hình autoencoder
autoencoder.save("Models/autoencoder_model.h5")

In [16]:
import pickle

# Lưu lại ngưỡng
with open("Models/threshold.pkl", "wb") as f:
    pickle.dump(threshold, f)

In [20]:
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError

autoencoder = load_model(
    "Models/autoencoder_model.h5",
    custom_objects={"mse": MeanSquaredError()}
)

# Tải lại ngưỡng
with open("Models/threshold.pkl", "rb") as f:
    threshold = pickle.load(f)

In [21]:
autoencoder.compile(optimizer="adadelta", loss="mse")
autoencoder.summary()

# Fit the model
history = autoencoder.fit(
    X_train,
    X_train,
    batch_size=64,
    epochs=10,
    verbose=0,
    validation_split=0.15,
    callbacks=[TqdmCallback(), EarlyStopping(patience=3)],
)

# Predict reconstruction errors for the training set
reconstructions = autoencoder.predict(X_train)
train_loss = tf.keras.losses.mae(reconstructions, X_train).numpy()
threshold = np.mean(train_loss) + np.std(train_loss)
print("Threshold: ", threshold)

# Predict reconstruction errors for the test set
reconstructions = autoencoder.predict(X_test)
test_loss = tf.keras.losses.mae(reconstructions, X_test).numpy()
preds = predict(autoencoder, X_test, threshold)

accuracy, precision, recall, f1 = print_stats(preds, y_test)
conf_matrix = confusion_matrix(y_test, preds)
print(conf_matrix)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Encoding_1 (Dense)              │ (None, 4)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Latent (Dense)                  │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Decoding_2 (Dense)              │ (None, 4)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 5)              │            25 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71 (284.00 B)

 Trainable params: 71 (284.00 B)

 Non-trainable params: 0 (0.00 B)

0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

4579/4579 ━━━━━━━━━━━━━━━━━━━━ 4s 922us/step
Threshold:  0.03264762728177069
1145/1145 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
1145/1145 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Accuracy = 0.9986896344626136
Precision = 0.99983484268766
Recall = 0.9988450433108759
F1 score = 0.9993396979117947
[[  260     6]
 [   42 36323]]
